# Bonus

🎯 84 feature’dan oluşan tam `ML_Houses_dataset.csv` dataset’iyle [buradan ulaşarak](https://d32aokrjazspmn.cloudfront.net/materials/ML_Houses_dataset.csv) serbestçe çalışabilirsiniz!

- Feature’ları inceleyin
- Uygun şekilde preprocess edin ve encode edin
- Feature engineering için beyin fırtınası yapın
- Bunları modelinize ekleyin
- Feature selection uygulayın

👇 Dosyayı yerel olarak `data` klasörüne kaydedin ve buradan içe aktarın.

In [1]:
import pandas as pd
import os
import urllib.request
import ssl

# 1. 'data' adında bir klasör oluşturalım (eğer henüz yoksa)
if not os.path.exists('data'):
    os.makedirs('data')

# 2. Veri linki ve kaydedeceğimiz yer
url = "https://d32aokrjazspmn.cloudfront.net/materials/ML_Houses_dataset.csv"
file_path = "data/ML_Houses_dataset.csv"

# 3. SSL sertifika hatasını atlamak için hazırlık (Eski dostumuz!)
ssl_context = ssl._create_unverified_context()

# 4. Dosyayı indirip 'data' klasörüne kaydedelim 
# (Eğer daha önce indirdiysek tekrar indirmemesi için if bloğu koyuyoruz)
if not os.path.exists(file_path):
    print("Veri internetten indiriliyor ve 'data' klasörüne kaydediliyor...")
    response = urllib.request.urlopen(url, context=ssl_context)
    with open(file_path, 'wb') as f:
        f.write(response.read())
    print("İndirme tamamlandı!")

# 5. Veriyi yerel klasörden okuyalım
df = pd.read_csv(file_path)

# Verinin boyutunu (satır ve sütun sayısı) ve ilk 5 satırını görelim
print("\nVeri Seti Boyutu:", df.shape)
display(df.head())

Veri internetten indiriliyor ve 'data' klasörüne kaydediliyor...
İndirme tamamlandı!

Veri Seti Boyutu: (1760, 85)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


ℹ️ Dataset’in açıklamasına mutlaka [buradan](https://drive.google.com/file/d/1qLxeQXufW_-KHOckpUweLPhitzjnP7H3/view?usp=sharing) referans verin.

Verideki eksiklikleri (Missing Values) tespit etmek

In [2]:
# Sütunlardaki eksik değerlerin oranını (%) büyükten küçüğe doğru sıralayarak görelim
missing_percentages = (df.isnull().sum() / len(df)) * 100
missing_percentages = missing_percentages[missing_percentages > 0].sort_values(ascending=False)

print("--- Eksik Verisi Olan Sütunlar ve Oranları (%) ---")
print(missing_percentages)

# Ayrıca hedef değişkenimiz 'Pesos' (Fiyat) mu, 'SalePrice' mı kontrol edelim
print("\nVeri setindeki sütunlar:", df.columns.tolist())

--- Eksik Verisi Olan Sütunlar ve Oranları (%) ---
WallMat         99.715909
PoolQC          99.488636
MiscFeature     96.250000
Alley           93.636364
Fence           80.568182
MasVnrType      60.340909
FireplaceQu     46.988636
LotFrontage     17.556818
GarageType       5.284091
GarageYrBlt      5.284091
GarageFinish     5.284091
GarageQual       5.284091
GarageCond       5.284091
BsmtFinType2     2.556818
BsmtExposure     2.500000
BsmtCond         2.443182
BsmtFinType1     2.443182
BsmtQual         2.443182
Pesos            0.738636
RoofSurface      0.568182
MasVnrArea       0.511364
Electrical       0.056818
dtype: float64

Veri setindeki sütunlar: ['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrT

In [3]:
# 1. Gereksiz ve sızıntı (leakage) yapabilecek sütunları çöpe atıyoruz
cols_to_drop = ['Id', 'WallMat', 'Pesos']
df = df.drop(columns=cols_to_drop)

# 2. Sözlüğe göre NA değerinin "Yok" (None) anlamına geldiği kategorik sütunlar
meaningful_na_cols = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtFinType2', 'BsmtExposure', 'BsmtCond', 'BsmtFinType1', 'BsmtQual'
]

# Bu sütunlardaki boşlukları 'None' kelimesiyle dolduruyoruz
for col in meaningful_na_cols:
    df[col] = df[col].fillna('None')

# 3. Sayısal eksik verileri medyan (ortanca) değer ile dolduruyoruz
num_cols_with_na = ['LotFrontage', 'MasVnrArea', 'GarageYrBlt', 'RoofSurface']
for col in num_cols_with_na:
    df[col] = df[col].fillna(df[col].median())

# 4. Kalan az sayıdaki kategorik eksiklikleri en çok tekrar eden değer (Mode) ile dolduruyoruz
cat_cols_with_na = ['MasVnrType', 'Electrical']
for col in cat_cols_with_na:
    df[col] = df[col].fillna(df[col].mode()[0])

# Son bir kontrol: Veri setimizde hiç eksik (NaN) değer kaldı mı?
print("Kalan toplam eksik (NaN) hücre sayısı:", df.isnull().sum().sum())

Kalan toplam eksik (NaN) hücre sayısı: 0


In [4]:
# 1. Toplam Yaşam Alanı (Total Living Area)
# Bodrum, 1. kat ve 2. kat alanlarını toplayarak evin gerçek boyutunu buluyoruz
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

# 2. Evin Yaşı (House Age)
# Satıldığı yıldan yapıldığı yılı çıkararak evin kaç yıllık olduğunu buluyoruz
df['HouseAge'] = df['YrSold'] - df['YearBuilt']

# 3. Tadilattan Sonra Geçen Süre (Years Since Remodel)
# Tadilatın etkisini ölçmek için
df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']

# 4. Toplam Banyo Sayısı (Total Bathrooms)
# Yarım banyoları 0.5 sayarak evin toplam banyo kapasitesini hesaplıyoruz
df['TotalBath'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])

# Kontrol edelim, yeni sütunlarımız eklendi mi?
print("Yeni sütunlar eklendi. Yeni sütun sayısı:", df.shape[1])
display(df[['TotalSF', 'HouseAge', 'TotalBath', 'SalePrice']].head())

Yeni sütunlar eklendi. Yeni sütun sayısı: 86


,TotalSF,HouseAge,TotalBath,SalePrice
0,2566,5,3.5,208500
1,2524,31,2.5,181500
2,2706,7,3.5,223500
3,2473,91,2.0,140000
4,3343,8,3.5,250000


**Feature Engineering**

Yeni Özellikler Türetme

In [5]:
# 1. Toplam Yaşam Alanı (Total Living Area)
# Bodrum, 1. kat ve 2. kat alanlarını toplayarak evin gerçek boyutunu buluyoruz
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

# 2. Evin Yaşı (House Age)
# Satıldığı yıldan yapıldığı yılı çıkararak evin kaç yıllık olduğunu buluyoruz
df['HouseAge'] = df['YrSold'] - df['YearBuilt']

# 3. Tadilattan Sonra Geçen Süre (Years Since Remodel)
# Tadilatın etkisini ölçmek için
df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']

# 4. Toplam Banyo Sayısı (Total Bathrooms)
# Yarım banyoları 0.5 sayarak evin toplam banyo kapasitesini hesaplıyoruz
df['TotalBath'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])

# Kontrol edelim, yeni sütunlarımız eklendi mi?
print("Yeni sütunlar eklendi. Yeni sütun sayısı:", df.shape[1])
display(df[['TotalSF', 'HouseAge', 'TotalBath', 'SalePrice']].head())

Yeni sütunlar eklendi. Yeni sütun sayısı: 86


,TotalSF,HouseAge,TotalBath,SalePrice
0,2566,5,3.5,208500
1,2524,31,2.5,181500
2,2706,7,3.5,223500
3,2473,91,2.0,140000
4,3343,8,3.5,250000


Kategorik Verileri Sayısallaştırma (Encoding)

In [6]:
# Tüm kategorik sütunları otomatik seçip sayılara çeviriyoruz
df_encoded = pd.get_dummies(df, drop_first=True)

print("Encoding sonrası toplam sütun sayısı:", df_encoded.shape[1])

Encoding sonrası toplam sütun sayısı: 266


Feature Selection (Korelasyon Analizi)

In [7]:
# 'SalePrice' ile en yüksek korelasyona sahip ilk 20 özelliği görelim
top_correlations = df_encoded.corr()['SalePrice'].sort_values(ascending=False).head(21)
print("--- SalePrice ile En Çok İlişkili Özellikler ---")
print(top_correlations)

# En önemli 15 özelliği seçelim (SalePrice kendisi dahil)
top_features = top_correlations.index.tolist()
df_final = df_encoded[top_features]

--- SalePrice ile En Çok İlişkili Özellikler ---
SalePrice               1.000000
OverallQual             0.792108
TotalSF                 0.762620
GrLivArea               0.699717
GarageCars              0.638577
TotalBath               0.620250
GarageArea              0.619462
TotalBsmtSF             0.604485
1stFlrSF                0.603921
FullBath                0.558821
TotRmsAbvGrd            0.539205
YearBuilt               0.518304
YearRemodAdd            0.505618
MasVnrArea              0.498908
Foundation_PConc        0.492409
GarageYrBlt             0.468582
Fireplaces              0.467854
ExterQual_Gd            0.434304
BsmtFinType1_GLQ        0.422445
Neighborhood_NridgHt    0.406248
BsmtFinSF1              0.368763
Name: SalePrice, dtype: float64


Nihai Modelleme (Random Forest)

In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# 1. Veriyi özellikler (X) ve hedef (y) olarak ayıralım
X = df_final.drop(columns=['SalePrice'])
y = df_final['SalePrice']

# 2. Eğitim ve test setlerine ayıralım
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Modeli kuralım ve eğitelim
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 4. Performansı ölçelim (R-squared skoru)
y_pred = rf_model.predict(X_test)
score = r2_score(y_test, y_pred)

print(f"Random Forest R2 Skoru: {score:.4f}")

Random Forest R2 Skoru: 0.9202
